
# QML-SleepNet — GUIDE-ONLY Stage 01→03 MetricMax Rebuild v1

**Purpose:** rebuild only the first three stages of the supervisor's supplied pipeline while reusing valid existing work.

This notebook deliberately does **not** continue the previous STGCN/Stage15 final architecture. It:

1. audits the existing Apnea-ECG Stage-02 cache;
2. reuses the mature 379-feature physiological bank where valid;
3. computes only guide-required feature families that are genuinely absent;
4. keeps all **x01–x35 labels inaccessible** during construction/model selection;
5. performs a subject/group-aware learn-set sanity benchmark;
6. creates the Stage-03 full guide feature bank for the next **128→64→32→8 classical-to-quantum bridge** notebook.

### Important ground-truth boundary
The Apnea-ECG minute annotations are **A/N (apnea / normal)**. AHI is therefore **not used as an input feature**, because deriving it from the reference apnea labels would leak the target. Predicted AHI belongs in Stage 09 evaluation.

### Output root
`/content/drive/MyDrive/QML_SleepNet/outputs/GUIDE_EXACT_METRICMAX/03_feature_bank_v1/`


In [ ]:

# Cell 1 — environment + Drive
!pip -q install wfdb PyWavelets xgboost

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, math, hashlib, warnings, time
import numpy as np
import pandas as pd
import scipy.signal as sps
import scipy.stats as st
from numpy.lib.stride_tricks import sliding_window_view
import pywt
import wfdb

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

PROJECT = Path("/content/drive/MyDrive/QML_SleepNet")
STAGE02 = PROJECT / "data" / "processed" / "stage02_preprocessed"
RAW = PROJECT / "data" / "raw" / "apnea_ecg"
OUT = PROJECT / "outputs" / "GUIDE_EXACT_METRICMAX" / "03_feature_bank_v1"
REC_OUT = OUT / "records"
REPORT = OUT / "reports"
for p in [OUT, REC_OUT, REPORT]:
    p.mkdir(parents=True, exist_ok=True)

FS = 100
EPOCH_SEC = 60
EPOCH_SAMPLES = FS * EPOCH_SEC
LEARN = [f"a{i:02d}" for i in range(1,21)] + [f"b{i:02d}" for i in range(1,6)] + [f"c{i:02d}" for i in range(1,11)]
TEST = [f"x{i:02d}" for i in range(1,36)]
ALL = LEARN + TEST
RESP_RECORDS = {"a01","a02","a03","a04","b01","c01","c02","c03"}

print("Project:", PROJECT)
print("Stage02 exists:", STAGE02.exists())
print("Output:", OUT)



## Cell 2 — locate the strongest reusable Stage-03 cache

The existing project already contains a later 379-feature bank. We reuse it only as a **label-free feature cache**. No old classifier, STGCN embedding, PCA reduction, threshold or previous final-model decision is inherited.


In [ ]:

# Cell 2 — robustly locate the existing 379-feature cache
SEARCH_ROOTS = [
    PROJECT / "outputs" / "guide_pipeline" / "15A_feature_bank_v1_4_3",
    PROJECT / "outputs" / "guide_pipeline" / "15A_feature_bank_v1_4_2",
    PROJECT / "outputs" / "guide_pipeline" / "15A_feature_bank_v1_4_1",
    PROJECT / "outputs" / "guide_pipeline" / "15A_feature_bank_v1_4_0",
    PROJECT / "outputs" / "guide_pipeline" / "15A_feature_bank_v1_2",
]

REQUIRED = {
    "feature_values_float32.npz",
    "feature_target_index.csv",
    "feature_metadata.csv",
    "feature_schema.json",
}

def valid_bank_dir(p: Path) -> bool:
    return p.is_dir() and all((p / x).exists() for x in REQUIRED)

bank_candidates = []
for root in SEARCH_ROOTS:
    if root.exists():
        if valid_bank_dir(root):
            bank_candidates.append(root)
        for p in root.rglob("*"):
            if valid_bank_dir(p):
                bank_candidates.append(p)

if not bank_candidates:
    raise FileNotFoundError(
        "Could not find an executed feature bank containing: " + ", ".join(sorted(REQUIRED))
    )

# Prefer the largest/newest valid bank.
bank_candidates = sorted(
    set(bank_candidates),
    key=lambda p: (
        (p / "feature_values_float32.npz").stat().st_size,
        (p / "feature_values_float32.npz").stat().st_mtime,
    ),
    reverse=True,
)
BASE = bank_candidates[0]
print("Reusable feature bank:", BASE)

z = np.load(BASE / "feature_values_float32.npz", allow_pickle=False)
X_base = np.asarray(z["X"], dtype=np.float32)
uids_base = np.asarray(z["uids"]).astype(str)
index = pd.read_csv(BASE / "feature_target_index.csv")
meta_base = pd.read_csv(BASE / "feature_metadata.csv")
schema = json.loads((BASE / "feature_schema.json").read_text())

assert len(index) == len(X_base) == len(uids_base)
assert index["uid"].astype(str).tolist() == uids_base.tolist()
assert X_base.shape[1] == len(meta_base)
assert schema.get("n_features", X_base.shape[1]) == X_base.shape[1]

print("Base X:", X_base.shape)
print("Index:", index.shape)
print("Records:", index["record_name"].nunique())
print("Base features:", X_base.shape[1])
print("Overall missing fraction:", float(np.isnan(X_base).mean()))


In [ ]:

# Cell 3 — Stage-02 audit with a hard label boundary
def load_stage2(rec: str, labels: bool = False):
    if labels and rec in TEST:
        raise RuntimeError(f"REFUSED: {rec} is an official x-record; held-out labels must remain inaccessible.")
    p = STAGE02 / f"{rec}_preprocessed.npz"
    if not p.exists():
        raise FileNotFoundError(p)
    d = np.load(p, allow_pickle=False)
    needed = {"ecg_filtered", "r_peaks", "fs", "n_epochs"}
    miss = needed - set(d.files)
    if miss:
        raise RuntimeError(f"{rec}: missing keys {sorted(miss)}")
    out = {
        "ecg": np.asarray(d["ecg_filtered"], dtype=np.float32),
        "rpeaks": np.asarray(d["r_peaks"], dtype=np.int64),
        "fs": int(np.asarray(d["fs"]).item()),
        "n_epochs": int(np.asarray(d["n_epochs"]).item()),
        "resp": np.asarray(d["resp"], dtype=np.float32) if "resp" in d.files else None,
        "spo2": np.asarray(d["spo2"], dtype=np.float32) if "spo2" in d.files else None,
    }
    if labels:
        if "labels" not in d.files:
            raise RuntimeError(f"{rec}: requested learn labels but no labels key exists")
        out["labels"] = np.asarray(d["labels"]).astype(str)
    if out["fs"] != FS:
        raise RuntimeError(f"{rec}: fs={out['fs']} != {FS}")
    if len(out["ecg"]) != out["n_epochs"] * EPOCH_SAMPLES:
        raise RuntimeError(f"{rec}: ECG length is not n_epochs × 6000")
    if len(out["rpeaks"]) > 1 and np.any(np.diff(out["rpeaks"]) <= 0):
        raise RuntimeError(f"{rec}: R peaks are not strictly increasing")
    return out

audit_rows = []
for rec in ALL:
    d = load_stage2(rec, labels=False)
    bank_n = int((index["record_name"].astype(str) == rec).sum())
    audit_rows.append({
        "record_name": rec,
        "stage02_epochs": d["n_epochs"],
        "bank_rows": bank_n,
        "rows_delta": d["n_epochs"] - bank_n,
        "rpeaks": len(d["rpeaks"]),
        "resp_array_present": d["resp"] is not None,
        "spo2_array_present": d["spo2"] is not None,
    })

stage2_audit = pd.DataFrame(audit_rows)
stage2_audit.to_csv(REPORT / "stage02_vs_basebank_audit.csv", index=False)
display(stage2_audit)
print("Total Stage02 epochs:", int(stage2_audit.stage02_epochs.sum()))
print("Total reusable bank rows:", int(stage2_audit.bank_rows.sum()))


In [ ]:

# Cell 4 — exact guide coverage audit BEFORE augmentation
names = meta_base["feature"].astype(str).tolist()

def has_any(*tokens):
    low = [x.lower() for x in names]
    return any(any(t.lower() in n for n in low) for t in tokens)

coverage_before = {
    "Time-domain HRV": has_any("rr_mean","sdnn","rmssd","pnn"),
    "Frequency-domain HRV": has_any("vlf","lf_power","hf_power","spectral"),
    "Nonlinear HRV": has_any("sampen","dfa","poincare"),
    "CWT": has_any("cwt"),
    "ECG morphology": has_any("qrs","morphology"),
    "db4 wavelet": has_any("db4","wavelet"),
    "STFT": has_any("stft"),
    "P-wave morphology": has_any("p_wave","pwave"),
    "T-wave morphology": has_any("t_wave","twave"),
    "ST deviation": has_any("st_deviation","st_dev"),
    "Raw respiration feature": has_any("resp_rate","respiration_rate"),
    "SpO2 feature": has_any("spo2"),
    "Transfer entropy": has_any("transfer_entropy"),
    "PLV": has_any("plv","phase_lock"),
    "Granger index": has_any("granger_index","granger_causality"),
}
coverage_before_df = pd.DataFrame(
    [{"guide_item": k, "already_explicitly_present": v} for k,v in coverage_before.items()]
)
coverage_before_df.to_csv(REPORT / "guide_coverage_before.csv", index=False)
display(coverage_before_df)



## Cells 5–7 — compute only the guide gaps

The following augmentation is intentionally compact. It derives:

- NN20 / NN50 counts, HRV triangular index, context SDANN-like measure;
- approximate entropy, DFA α2 and recurrence quantification summaries;
- explicit QRS duration/amplitude, P-wave, T-wave and ST-segment summaries;
- db4 wavelet energies;
- STFT band/entropy/ridge summaries and instantaneous-frequency descriptors;
- for the eight records with respiratory channels: respiration rate, SpO₂ nadir/desaturation, thoraco-abdominal opposition and ECG-respiration coupling measures.

The existing 379 features remain untouched.


In [ ]:

# Cell 5 — reusable signal/HRV helpers
EPS = 1e-10

def safe_mean(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def safe_std(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return float(np.std(x, ddof=1)) if len(x) > 1 else np.nan

def rr_for_epoch(rpeaks, e, context=0):
    lo = max(0, (e-context) * EPOCH_SAMPLES)
    hi = (e+context+1) * EPOCH_SAMPLES
    # include a beat immediately before the window to preserve one boundary RR
    j0 = max(0, np.searchsorted(rpeaks, lo) - 1)
    j1 = np.searchsorted(rpeaks, hi)
    p = rpeaks[j0:j1] / FS
    rr = np.diff(p)
    rr = rr[(rr >= 0.30) & (rr <= 2.00)]
    return rr

def approx_entropy(x, m=2, r_ratio=0.2):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < max(12, m+2) or np.std(x) < EPS:
        return np.nan
    r = r_ratio * np.std(x)
    def phi(mm):
        w = sliding_window_view(x, mm)
        if len(w) < 2:
            return np.nan
        dist = np.max(np.abs(w[:,None,:] - w[None,:,:]), axis=2)
        c = np.mean(dist <= r, axis=1)
        return np.mean(np.log(np.maximum(c, EPS)))
    a, b = phi(m), phi(m+1)
    return float(a-b) if np.isfinite(a) and np.isfinite(b) else np.nan

def dfa_alpha(x, min_scale=16, max_scale=None):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 40:
        return np.nan
    if max_scale is None:
        max_scale = max(min_scale+2, min(64, n//4))
    if max_scale <= min_scale:
        return np.nan
    y = np.cumsum(x - np.mean(x))
    scales = np.unique(np.floor(np.geomspace(min_scale, max_scale, 6)).astype(int))
    vals = []
    for s in scales:
        k = n // s
        if k < 2:
            continue
        rms = []
        t = np.arange(s)
        for i in range(k):
            seg = y[i*s:(i+1)*s]
            coef = np.polyfit(t, seg, 1)
            detr = seg - np.polyval(coef, t)
            rms.append(np.sqrt(np.mean(detr*detr)))
        f = np.mean(rms)
        if f > 0:
            vals.append((s, f))
    if len(vals) < 3:
        return np.nan
    a = np.polyfit(np.log([v[0] for v in vals]), np.log([v[1] for v in vals]), 1)[0]
    return float(a)

def rqa_features(x, eps_ratio=0.20):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 15 or np.std(x) < EPS:
        return np.nan, np.nan, np.nan
    z = (x-np.mean(x))/(np.std(x)+EPS)
    R = np.abs(z[:,None]-z[None,:]) <= eps_ratio
    np.fill_diagonal(R, False)
    rr = float(R.mean())

    diag_points = 0
    diag_total = int(R.sum())
    for k in range(-len(x)+1, len(x)):
        if k == 0: 
            continue
        d = np.diag(R, k=k).astype(np.int8)
        if len(d) < 2: 
            continue
        runs = np.diff(np.r_[0, d, 0])
        starts = np.where(runs == 1)[0]
        ends = np.where(runs == -1)[0]
        lengths = ends - starts
        diag_points += int(lengths[lengths >= 2].sum())
    determinism = float(diag_points / diag_total) if diag_total else np.nan

    vert_points = 0
    for j in range(R.shape[1]):
        d = R[:,j].astype(np.int8)
        runs = np.diff(np.r_[0, d, 0])
        starts = np.where(runs == 1)[0]
        ends = np.where(runs == -1)[0]
        lengths = ends - starts
        vert_points += int(lengths[lengths >= 2].sum())
    laminarity = float(vert_points / diag_total) if diag_total else np.nan
    return rr, determinism, laminarity

def triangular_index(rr):
    rr = np.asarray(rr, float)
    rr = rr[np.isfinite(rr)]
    if len(rr) < 8:
        return np.nan
    bw = 1/128.0
    bins = np.arange(max(0.25, rr.min()-bw), min(2.25, rr.max()+2*bw), bw)
    if len(bins) < 3:
        return np.nan
    h, _ = np.histogram(rr, bins=bins)
    return float(len(rr) / max(h.max(), 1))

def normalized_wavelet_energies(x):
    x = np.asarray(x, float)
    x = np.nan_to_num(x, nan=0.0)
    x = x - np.mean(x)
    sd = np.std(x)
    if sd > EPS:
        x = x/sd
    coeffs = pywt.wavedec(x, "db4", level=5)
    en = np.array([np.sum(c*c) for c in coeffs], float)
    en /= (en.sum()+EPS)
    return en  # A5,D5,D4,D3,D2,D1

def morphology_features(ecg, rpeaks, e):
    lo = e*EPOCH_SAMPLES
    hi = (e+1)*EPOCH_SAMPLES
    rp = rpeaks[(rpeaks >= lo+25) & (rpeaks < hi-45)]
    if len(rp) < 4:
        return [np.nan]*6
    beats = []
    for r in rp:
        seg = ecg[r-25:r+45]
        if len(seg)==70 and np.isfinite(seg).all():
            beats.append(seg)
    if len(beats) < 4:
        return [np.nan]*6
    B = np.asarray(beats, float)
    template = np.median(B, axis=0)
    # R is index 25
    pr_base = np.median(template[5:13])          # -200..-120 ms
    p_amp = np.max(np.abs(template[5:17]-pr_base))  # approx -200..-80
    qrs = template[17:36] - pr_base            # approx -80..+110
    qrs_amp = np.max(qrs)-np.min(qrs)
    thr = 0.10 * max(np.max(np.abs(qrs)), EPS)
    active = np.where(np.abs(qrs) >= thr)[0]
    qrs_dur_ms = (active[-1]-active[0]+1)*10.0 if len(active) else np.nan
    st_dev = np.median(template[33:37]) - pr_base  # approx +80..+120 ms
    t_amp = np.max(np.abs(template[37:65]-pr_base)) # approx +120..+400 ms
    cors = []
    for b in B:
        if np.std(b)>EPS and np.std(template)>EPS:
            cors.append(np.corrcoef(b,template)[0,1])
    template_corr = safe_mean(cors)
    return [qrs_dur_ms, qrs_amp, p_amp, t_amp, st_dev, template_corr]

def stft_features(x):
    x = np.asarray(x, float)
    if len(x) != EPOCH_SAMPLES:
        return [np.nan]*9
    f,t,Z = sps.stft(x, fs=FS, nperseg=128, noverlap=64, boundary=None)
    P = np.abs(Z)**2
    def band(a,b):
        m=(f>=a)&(f<b)
        return float(P[m].sum()/(P.sum()+EPS))
    bands=[band(.5,3),band(3,8),band(8,15),band(15,25),band(25,40)]
    ps = P.sum(axis=1)
    ps = ps/(ps.sum()+EPS)
    ent = float(-np.sum(ps*np.log(ps+EPS))/np.log(max(len(ps),2)))
    valid=(f>=.5)&(f<=40)
    fv=f[valid]; Pv=P[valid]
    ridge = fv[np.argmax(Pv,axis=0)] if Pv.size else np.array([])
    ridge_med=safe_mean(ridge)
    ridge_iqr=float(np.subtract(*np.percentile(ridge,[75,25]))) if len(ridge) else np.nan
    # Hilbert instantaneous frequency after ECG-band smoothing
    sos=sps.butter(3,[0.5,15],btype="band",fs=FS,output="sos")
    xf=sps.sosfiltfilt(sos,x)
    ph=np.unwrap(np.angle(sps.hilbert(xf)))
    inst=np.diff(ph)*FS/(2*np.pi)
    inst=inst[np.isfinite(inst)&(inst>=0)&(inst<=40)]
    return bands+[ent,ridge_med,ridge_iqr,safe_mean(inst)]


In [ ]:

# Cell 6 — respiration / cross-modal helpers for the 8 multimodal records
def align_len(x, n):
    x=np.asarray(x,float)
    if len(x)>=n:
        return x[:n]
    if len(x)==0:
        return np.full(n,np.nan)
    return np.pad(x,(0,n-len(x)),constant_values=np.nan)

def clean_interp(x):
    x=np.asarray(x,float)
    ok=np.isfinite(x)
    if ok.sum()<2:
        return np.full_like(x,np.nan)
    return np.interp(np.arange(len(x)),np.flatnonzero(ok),x[ok])

def respiration_rate(sig, fs=FS):
    sig=clean_interp(sig)
    if not np.isfinite(sig).any() or len(sig)<fs*20:
        return np.nan
    sos=sps.butter(3,[0.08,0.7],btype="band",fs=fs,output="sos")
    y=sps.sosfiltfilt(sos,sig)
    distance=int(fs*1.0)
    peaks,_=sps.find_peaks(y,distance=distance,prominence=max(np.std(y)*0.15,EPS))
    return float(len(peaks)*60/(len(y)/fs))

def phase_locking_value(x,y,fs=4.0):
    if len(x)<32 or len(y)!=len(x):
        return np.nan
    sos=sps.butter(3,[0.1,0.4],btype="band",fs=fs,output="sos")
    try:
        a=sps.sosfiltfilt(sos,clean_interp(x))
        b=sps.sosfiltfilt(sos,clean_interp(y))
        d=np.angle(sps.hilbert(a))-np.angle(sps.hilbert(b))
        return float(np.abs(np.mean(np.exp(1j*d))))
    except Exception:
        return np.nan

def transfer_entropy_discrete(x,y,bins=4):
    x=clean_interp(x); y=clean_interp(y)
    if len(x)<20 or len(y)!=len(x):
        return np.nan
    def disc(a):
        q=np.unique(np.quantile(a,np.linspace(0,1,bins+1)))
        if len(q)<3:
            return np.zeros(len(a),int)
        return np.digitize(a,q[1:-1],right=False)
    X=disc(x); Y=disc(y)
    # TE X->Y: I(Y_t ; X_{t-1} | Y_{t-1})
    counts={}
    for yt,yp,xp in zip(Y[1:],Y[:-1],X[:-1]):
        counts[(yt,yp,xp)]=counts.get((yt,yp,xp),0)+1
    n=max(sum(counts.values()),1)
    te=0.0
    for (yt,yp,xp),c in counts.items():
        pxyz=c/n
        cyx=sum(v for (a,b,d),v in counts.items() if b==yp and d==xp)
        cy=sum(v for (a,b,d),v in counts.items() if a==yt and b==yp)
        cy0=sum(v for (a,b,d),v in counts.items() if b==yp)
        if min(cyx,cy,cy0)>0:
            te += pxyz*np.log((c/cyx)/(cy/cy0)+EPS)
    return float(max(te,0.0))

def granger_index(x,y,order=4):
    # predictive-directionality score X -> Y, no causal claim
    x=clean_interp(x); y=clean_interp(y)
    n=min(len(x),len(y))
    x=x[:n]; y=y[:n]
    if n<50:
        return np.nan
    rows=[]; target=[]
    for t in range(order,n):
        yl=y[t-order:t][::-1]
        xl=x[t-order:t][::-1]
        rows.append((yl,xl)); target.append(y[t])
    Y=np.asarray(target)
    R=np.asarray([r[0] for r in rows])
    F=np.asarray([np.r_[r[0],r[1]] for r in rows])
    R=np.c_[np.ones(len(R)),R]
    F=np.c_[np.ones(len(F)),F]
    br=np.linalg.lstsq(R,Y,rcond=None)[0]
    bf=np.linalg.lstsq(F,Y,rcond=None)[0]
    vr=np.mean((Y-R@br)**2)+EPS
    vf=np.mean((Y-F@bf)**2)+EPS
    return float(np.log(vr/vf))

def resample_to_4hz(sig, start, end):
    seg=clean_interp(sig[start:end])
    if not np.isfinite(seg).any():
        return np.array([])
    n=max(int(round(len(seg)/FS*4)),2)
    return sps.resample(seg,n)

def edr_from_ramps(ecg,rpeaks,start,end):
    rp=rpeaks[(rpeaks>=start)&(rpeaks<end)]
    if len(rp)<5:
        return np.array([])
    amp=ecg[rp].astype(float)
    tt=(rp-start)/FS
    grid=np.arange(0,(end-start)/FS,0.25)
    if len(grid)<2:
        return np.array([])
    return np.interp(grid,tt,amp,left=amp[0],right=amp[-1])

def cross_modal_context(ecg,rpeaks,resp,e,n_epochs):
    e0=max(0,e-2); e1=min(n_epochs,e+3)
    start=e0*EPOCH_SAMPLES; end=e1*EPOCH_SAMPLES
    edr=edr_from_ramps(ecg,rpeaks,start,end)
    rrsp=resample_to_4hz(resp,start,end)
    n=min(len(edr),len(rrsp))
    if n<64:
        return [np.nan]*5
    edr=edr[:n]; rrsp=rrsp[:n]
    f,cxy=sps.coherence(clean_interp(edr),clean_interp(rrsp),fs=4,nperseg=min(128,n))
    m=(f>=.1)&(f<=.4)
    coh=float(np.mean(cxy[m])) if m.any() else np.nan
    plv=phase_locking_value(edr,rrsp,4.0)
    te=transfer_entropy_discrete(edr,rrsp)
    gi=granger_index(edr,rrsp,order=4)
    # bivariate spectral coupling ratio
    f2,Pxy=sps.csd(clean_interp(edr),clean_interp(rrsp),fs=4,nperseg=min(128,n))
    m2=(f2>=.1)&(f2<=.4)
    ratio=float(np.sum(np.abs(Pxy[m2]))/(np.sum(np.abs(Pxy))+EPS)) if m2.any() else np.nan
    return [coh,te,plv,gi,ratio]

def raw_resp_channels(rec, n_samples):
    if rec not in RESP_RECORDS:
        return None
    p=str(RAW/f"{rec}r")
    try:
        sig, fields = wfdb.rdsamp(p)
        names=[str(x) for x in fields.get("sig_name",[])]
        d={n:align_len(sig[:,i],n_samples) for i,n in enumerate(names)}
        return d
    except Exception as e:
        print(f"{rec}: raw standalone respiration read failed; using Stage02 selected respiration only:", repr(e))
        return None


In [ ]:

# Cell 7 — per-record guide-gap augmentation with checkpointing
AUG_NAMES = [
    "guide_nn20_count","guide_nn50_count","guide_hrv_triangular_index",
    "guide_sdann_5min","guide_approx_entropy","guide_dfa_alpha2",
    "guide_rqa_recurrence_rate","guide_rqa_determinism","guide_rqa_laminarity",
    "guide_qrs_duration_ms","guide_qrs_amplitude","guide_p_wave_amplitude",
    "guide_t_wave_amplitude","guide_st_deviation","guide_ecg_template_similarity",
    "guide_db4_A5_energy","guide_db4_D5_energy","guide_db4_D4_energy",
    "guide_db4_D3_energy","guide_db4_D2_energy","guide_db4_D1_energy",
    "guide_stft_e_0p5_3","guide_stft_e_3_8","guide_stft_e_8_15",
    "guide_stft_e_15_25","guide_stft_e_25_40","guide_stft_entropy",
    "guide_stft_ridge_median_hz","guide_stft_ridge_iqr_hz","guide_instantaneous_freq_mean_hz",
    "guide_resp_available","guide_resp_rate_bpm","guide_spo2_nadir",
    "guide_spo2_desaturation_depth","guide_thoraco_abdominal_opposition",
    "guide_ecg_resp_coherence","guide_transfer_entropy_ecg_to_resp",
    "guide_ecg_resp_plv","guide_granger_predictive_index_ecg_to_resp",
    "guide_bivariate_spectral_ratio",
]

def augment_record(rec):
    outp=REC_OUT/f"{rec}_guide_augment.npz"
    if outp.exists():
        q=np.load(outp,allow_pickle=False)
        if list(np.asarray(q["names"]).astype(str))==AUG_NAMES:
            return np.asarray(q["X"],dtype=np.float32), np.asarray(q["epochs"],dtype=int)

    d=load_stage2(rec,labels=False)
    ecg=d["ecg"]; rp=d["rpeaks"]; n_epochs=d["n_epochs"]
    rawch=raw_resp_channels(rec,len(ecg))
    resp=d["resp"]
    spo2=d["spo2"]

    # Choose a respiration waveform for rate/cross-modal analysis.
    if rawch:
        resp_candidates=[]
        for key in ["Resp N","Resp C","Resp A"]:
            if key in rawch:
                resp_candidates.append(rawch[key])
        resp_sig=resp_candidates[0] if resp_candidates else resp
        chest=rawch.get("Resp C")
        abd=rawch.get("Resp A")
        if "SpO2" in rawch:
            spo2_sig=rawch["SpO2"]
        else:
            spo2_sig=spo2
    else:
        resp_sig=resp
        chest=abd=None
        spo2_sig=spo2

    # precompute one-minute RR means for context SDANN
    rr_means=np.full(n_epochs,np.nan)
    for e in range(n_epochs):
        rr=rr_for_epoch(rp,e,context=0)
        rr_means[e]=safe_mean(rr)

    rows=[]
    for e in range(n_epochs):
        rr=rr_for_epoch(rp,e,context=0)
        rr5=rr_for_epoch(rp,e,context=2)

        dr=np.abs(np.diff(rr)) if len(rr)>1 else np.array([])
        nn20=float(np.sum(dr>.020)) if len(dr) else np.nan
        nn50=float(np.sum(dr>.050)) if len(dr) else np.nan
        tri=triangular_index(rr)

        c0=max(0,e-2); c1=min(n_epochs,e+3)
        mm=rr_means[c0:c1]
        sdann=safe_std(mm)
        apen=approx_entropy(rr5)
        a2=dfa_alpha(rr5,min_scale=16,max_scale=min(64,max(18,len(rr5)//4))) if len(rr5)>=40 else np.nan
        rqa_rr,rqa_det,rqa_lam=rqa_features(rr5)

        morph=morphology_features(ecg,rp,e)
        seg=ecg[e*EPOCH_SAMPLES:(e+1)*EPOCH_SAMPLES]
        wav=normalized_wavelet_energies(seg)
        stft=stft_features(seg)

        available=1.0 if rec in RESP_RECORDS and resp_sig is not None else 0.0
        if available:
            rs=resp_sig[e*EPOCH_SAMPLES:(e+1)*EPOCH_SAMPLES]
            rr_bpm=respiration_rate(rs)
            if spo2_sig is not None:
                ss=spo2_sig[e*EPOCH_SAMPLES:(e+1)*EPOCH_SAMPLES]
                ss=ss[np.isfinite(ss)]
                nad=float(np.min(ss)) if len(ss) else np.nan
                des=float(np.median(ss)-np.min(ss)) if len(ss) else np.nan
            else:
                nad=des=np.nan

            if chest is not None and abd is not None:
                cs=clean_interp(chest[e*EPOCH_SAMPLES:(e+1)*EPOCH_SAMPLES])
                ab=clean_interp(abd[e*EPOCH_SAMPLES:(e+1)*EPOCH_SAMPLES])
                if np.std(cs)>EPS and np.std(ab)>EPS:
                    c=np.corrcoef(cs,ab)[0,1]
                    opposition=float(max(0.0,-c))
                else:
                    opposition=np.nan
            else:
                opposition=np.nan
            cm=cross_modal_context(ecg,rp,resp_sig,e,n_epochs)
        else:
            rr_bpm=nad=des=opposition=np.nan
            cm=[np.nan]*5

        row = [
            nn20,nn50,tri,sdann,apen,a2,rqa_rr,rqa_det,rqa_lam,
            *morph,*wav,*stft,
            available,rr_bpm,nad,des,opposition,*cm
        ]
        if len(row)!=len(AUG_NAMES):
            raise RuntimeError((rec,e,len(row),len(AUG_NAMES)))
        rows.append(row)

    X=np.asarray(rows,dtype=np.float32)
    epochs=np.arange(n_epochs,dtype=np.int32)
    tmp=outp.with_suffix(".tmp.npz")
    np.savez_compressed(tmp,X=X,epochs=epochs,names=np.asarray(AUG_NAMES,dtype="U96"))
    os.replace(tmp,outp)
    print(f"{rec}: augmented {X.shape}")
    return X,epochs

t0=time.time()
for i,rec in enumerate(ALL,1):
    augment_record(rec)
    if i%5==0:
        print(f"[{i}/{len(ALL)}] elapsed {(time.time()-t0)/60:.1f} min")
print("Augmentation complete.")


In [ ]:

# Cell 8 — merge augmentation with the existing 379-feature rows by record+epoch
base_names=meta_base["feature"].astype(str).tolist()
aug_map={}
for rec in ALL:
    q=np.load(REC_OUT/f"{rec}_guide_augment.npz",allow_pickle=False)
    Xr=np.asarray(q["X"],dtype=np.float32)
    er=np.asarray(q["epochs"],dtype=int)
    aug_map[rec]=(Xr,{int(e):i for i,e in enumerate(er)})

X_aug=np.full((len(index),len(AUG_NAMES)),np.nan,dtype=np.float32)
for i,row in index[["record_name","epoch_idx"]].iterrows():
    rec=str(row.record_name); e=int(row.epoch_idx)
    Xr,emap=aug_map[rec]
    if e in emap:
        X_aug[i]=Xr[emap[e]]

X_full=np.concatenate([X_base,X_aug],axis=1)
FULL_NAMES=base_names+AUG_NAMES
assert X_full.shape[1]==len(FULL_NAMES)
assert len(set(FULL_NAMES))==len(FULL_NAMES)

# We preserve NaNs; imputation must be fitted only on training partitions.
np.savez_compressed(
    OUT/"guide_stage03_full_feature_bank.npz",
    X=X_full.astype(np.float32),
    uids=uids_base.astype("U128"),
    feature_names=np.asarray(FULL_NAMES,dtype="U128")
)
index.to_csv(OUT/"guide_stage03_target_index.csv",index=False)

aug_meta=pd.DataFrame({
    "feature":AUG_NAMES,
    "group":[
        "hrv_time","hrv_time","hrv_time","hrv_time","hrv_nonlinear","hrv_nonlinear",
        "hrv_nonlinear","hrv_nonlinear","hrv_nonlinear",
        "ecg_morphology","ecg_morphology","ecg_morphology","ecg_morphology","ecg_morphology","ecg_morphology",
        "wavelet_db4","wavelet_db4","wavelet_db4","wavelet_db4","wavelet_db4","wavelet_db4",
        "stft","stft","stft","stft","stft","stft","stft","stft","stft",
        "resp_quality","resp","spo2","spo2","resp","cross_modal","cross_modal","cross_modal","cross_modal","cross_modal"
    ],
    "source":"guide_gap_augmentation_v1",
})
meta_out=pd.concat(
    [meta_base.assign(source="reused_379_feature_cache"),aug_meta],
    ignore_index=True,sort=False
)
meta_out.to_csv(OUT/"guide_stage03_feature_metadata.csv",index=False)

print("Merged feature bank:",X_full.shape)
print("Base features:",len(base_names),"New exact-guide-gap features:",len(AUG_NAMES))
print("Merged missing fraction:",float(np.isnan(X_full).mean()))


In [ ]:

# Cell 9 — guide coverage AFTER augmentation
full_low=[x.lower() for x in FULL_NAMES]
def full_has(*tokens):
    return any(any(t.lower() in n for n in full_low) for t in tokens)

coverage_after = {
    "Time-domain HRV": full_has("rr_mean","sdnn","rmssd","guide_nn20","triangular"),
    "Frequency-domain HRV": full_has("vlf","lf_power","hf_power"),
    "Nonlinear HRV": full_has("sampen","guide_approx_entropy","guide_dfa_alpha2","guide_rqa"),
    "CWT": full_has("cwt"),
    "ECG morphology": full_has("guide_qrs_duration","guide_qrs_amplitude"),
    "P-wave morphology": full_has("guide_p_wave"),
    "T-wave morphology": full_has("guide_t_wave"),
    "ST deviation": full_has("guide_st_deviation"),
    "db4 wavelet": full_has("guide_db4"),
    "STFT": full_has("guide_stft"),
    "Respiration rate": full_has("guide_resp_rate"),
    "SpO2": full_has("guide_spo2"),
    "ECG-Resp coherence": full_has("guide_ecg_resp_coherence"),
    "Transfer entropy": full_has("guide_transfer_entropy"),
    "PLV": full_has("guide_ecg_resp_plv"),
    "Granger predictive index": full_has("guide_granger_predictive_index"),
    "Bivariate spectral ratio": full_has("guide_bivariate_spectral_ratio"),
}
cov=pd.DataFrame([{"guide_item":k,"present_after_rebuild":v} for k,v in coverage_after.items()])
cov.to_csv(REPORT/"guide_coverage_after.csv",index=False)
display(cov)
if not cov["present_after_rebuild"].all():
    raise RuntimeError("One or more intended guide Stage-03 feature families remain absent.")
print("STAGE-03 GUIDE COVERAGE CHECK: PASS")



## Cell 10 — learn-only, subject/group-aware sanity benchmark

This is **not** the official blind-test result. Its only purpose is to answer a practical question before investing in QML training:

> Does the rebuilt Stage-03 representation already live in a strong predictive regime on the 35 learning recordings?

`x01–x35` labels are never accessed here. `c05` and `c06` are forced into the same group.


In [ ]:

# Cell 10 — 5-fold group-aware XGBoost sanity benchmark
idx=index.copy()
rec=idx["record_name"].astype(str).to_numpy()
learn_mask=np.isin(rec,LEARN)

# IMPORTANT: y comes only from the existing target index for LEARN rows.
# The x-record rows are not touched by this benchmark.
y_raw=idx.loc[learn_mask,"y"].astype(str).to_numpy()
if not set(np.unique(y_raw)).issubset({"A","N","0","1","0.0","1.0"}):
    print("Observed learn labels:",np.unique(y_raw))
y=np.array([1 if str(v).upper()=="A" or str(v)=="1" or str(v)=="1.0" else 0 for v in y_raw],dtype=np.int8)
X=X_full[learn_mask]
r=rec[learn_mask]
groups=np.array(["c05_c06" if x in {"c05","c06"} else x for x in r])

cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
rows=[]
oof=np.full(len(y),np.nan)
for fold,(tr,va) in enumerate(cv.split(X,y,groups)):
    imp=SimpleImputer(strategy="median",add_indicator=True)
    Xt=imp.fit_transform(X[tr])
    Xv=imp.transform(X[va])

    clf=XGBClassifier(
        n_estimators=450,max_depth=5,learning_rate=0.04,
        subsample=0.88,colsample_bytree=0.88,min_child_weight=3,
        reg_lambda=2.0,reg_alpha=0.05,
        eval_metric="logloss",tree_method="hist",
        random_state=SEED+fold,n_jobs=-1
    )
    clf.fit(Xt,y[tr])
    p=clf.predict_proba(Xv)[:,1]
    pred=(p>=0.5).astype(int)
    oof[va]=p
    m={
        "fold":fold,
        "n_train":len(tr),"n_val":len(va),
        "accuracy":accuracy_score(y[va],pred),
        "balanced_accuracy":balanced_accuracy_score(y[va],pred),
        "f1":f1_score(y[va],pred),
        "auroc":roc_auc_score(y[va],p),
        "auprc":average_precision_score(y[va],p),
    }
    rows.append(m)
    print(m)

cvdf=pd.DataFrame(rows)
cvdf.to_csv(REPORT/"stage03_groupcv_xgb_sanity.csv",index=False)
pd.DataFrame({"uid":idx.loc[learn_mask,"uid"].astype(str),"y":y,"oof_prob":oof}).to_csv(
    REPORT/"stage03_groupcv_xgb_oof.csv",index=False
)
display(cvdf)
print("\nMEAN")
display(cvdf[["accuracy","balanced_accuracy","f1","auroc","auprc"]].mean().to_frame("mean").T)

# No hard 90% assertion: the experiment reports the truth.



## Cell 11 — freeze Stage-03 inputs for the next notebook

No ANOVA/mRMR/SHAP/PCA is fitted globally here. That is deliberate.

The next notebook will perform **ANOVA → mRMR → SHAP → PCA whitening → 128-D** *inside each learning fold*, evaluate the exact classical `128→64→32→8` bridge, select the strongest guide-permitted configuration, then refit that configuration on all 35 learning records for Stage 04. This prevents feature-selection leakage into cross-validation.


In [ ]:

# Cell 11 — manifest + hashes
def sha256_file(p,chunk=1<<20):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

bank_path=OUT/"guide_stage03_full_feature_bank.npz"
idx_path=OUT/"guide_stage03_target_index.csv"
meta_path=OUT/"guide_stage03_feature_metadata.csv"

manifest={
    "pipeline":"QML-SleepNet GUIDE-ONLY Stage01-03 MetricMax Rebuild v1",
    "dataset":"Apnea-ECG",
    "fs_hz":FS,
    "epoch_seconds":EPOCH_SEC,
    "learn_records":LEARN,
    "official_test_records":TEST,
    "official_test_labels_accessed_for_model_selection":False,
    "reused_base_feature_bank":str(BASE),
    "n_rows":int(len(index)),
    "n_base_features":int(len(base_names)),
    "n_added_guide_gap_features":int(len(AUG_NAMES)),
    "n_total_features":int(len(FULL_NAMES)),
    "stage03_feature_bank_sha256":sha256_file(bank_path),
    "stage03_target_index_sha256":sha256_file(idx_path),
    "stage03_metadata_sha256":sha256_file(meta_path),
    "ahi_input_feature_used":False,
    "task_b_osa_csa_mixed_ground_truth_created":False,
    "next":"fold-safe ANOVA→mRMR→SHAP→PCA128 + classical 128→64→32→8 bridge",
}
(OUT/"STAGE03_GUIDE_REBUILD_MANIFEST.json").write_text(json.dumps(manifest,indent=2))

print("="*100)
print("STAGE 01→03 GUIDE-ONLY REBUILD COMPLETE")
print("="*100)
print("Feature bank:",bank_path)
print("Rows × features:",X_full.shape)
print("Manifest:",OUT/"STAGE03_GUIDE_REBUILD_MANIFEST.json")
print("NEXT: Stage 03 selection + Stage 04 classical→quantum bridge.")
